# Which African Country Has the Most Languages?

This page uses the [TidyTuesday 2026-01-13 dataset](https://github.com/rfordatascience/tidytuesday/tree/main/data/2026/2026-01-13)
on popular languages of Africa, scraped from the Wikipedia page
*Languages of Africa*. Each row is one language–country pair, with the language's
family and its number of native speakers.

**Question:** which country in Africa has the largest number of spoken languages?

**Short answer:** in this dataset, **Cameroon** (95 languages). The rest of this page
checks whether that answer holds up, and what kind of languages drive it.

```{note} What this dataset counts
:class: dropdown
Africa has somewhere between 1,250 and 2,100+ native languages, but this dataset
only covers the **popular** languages listed in one Wikipedia table (about 500
languages). So "number of languages" here means *number of listed languages
linked to that country*, not every language spoken there. Code cells are folded;
click one to see the code.
```

## The data

In [1]:
import os

import matplotlib.pyplot as plt
import pandas as pd

url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-01-13/africa.csv"
local = "data/africa.csv"
raw = pd.read_csv(local if os.path.exists(local) else url)

# Clean: drop exact duplicate rows and merge two spellings of the same family
africa = raw.drop_duplicates().copy()
africa["family"] = africa["family"].replace({"Afro-Asiatic": "Afroasiatic"})

os.makedirs("images", exist_ok=True)
BLUE, ORANGE, GREY = "tab:blue", "tab:orange", "#b0b0b0"

pd.DataFrame({
    "value": [len(raw), raw.duplicated().sum(), len(africa),
              africa["language"].nunique(), africa["country"].nunique(),
              africa["family"].nunique()],
}, index=["Rows (raw)", "Exact duplicate rows", "Rows after cleaning",
          "Unique languages", "Countries", "Language families"])

,value
Rows (raw),796
Exact duplicate rows,34
Rows after cleaning,762
Unique languages,502
Countries,51
Language families,16


Two cleaning steps: the raw file has 34 exact duplicate rows, and the
Afroasiatic family is spelled two ways (`Afroasiatic` and `Afro-Asiatic`).

## 1. Counting languages per country

In [2]:
langs_per_country = (
    africa.groupby("country")["language"]
    .nunique()
    .sort_values(ascending=False)
    .rename("n_languages")
)
top_country = langs_per_country.index[0]
top10_names = langs_per_country.head(10).index

top10 = langs_per_country.head(10).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top10.index, top10.values,
        color=[ORANGE if c == top_country else BLUE for c in top10.index])
for y, v in enumerate(top10.values):
    ax.text(v + 1, y, str(v), va="center")
ax.set_xlabel("Number of unique languages in the dataset")
ax.set_title(f"{top_country} has the most languages in the dataset ({langs_per_country.max()})")
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(0, top10.max() * 1.12)
fig.savefig("images/languages_per_country.png", dpi=150, bbox_inches="tight")
plt.close(fig)

langs_per_country.head(10).to_frame()

,n_languages
country,
Cameroon,95
Congo,79
Nigeria,73
Sudan,40
Burkina Faso,36
Ghana,34
Namibia,24
Chad,24
South Sudan,23


```{figure} images/languages_per_country.png
:name: fig-langs-per-country
:alt: Horizontal bar chart of the ten African countries with the most languages in the dataset, with Cameroon at the top
:width: 85%

The ten countries with the most unique languages in the dataset. Cameroon leads
with 95, followed by "Congo" (79) and Nigeria (73).
```

## 2. How uneven is it?

In [3]:
top3 = langs_per_country.head(3).index
top3_langs = africa.loc[africa["country"].isin(top3), "language"].nunique()
share_top3 = top3_langs / africa["language"].nunique()
n_small = (langs_per_country <= 5).sum()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(langs_per_country, bins=range(0, 101, 5), color=BLUE, edgecolor="white")
label_offsets = {top3[0]: (0, 18), top3[1]: (14, 36), top3[2]: (-14, 18)}
for c in top3:
    ax.annotate(c, (langs_per_country[c], 1), xytext=label_offsets[c], textcoords="offset points",
                ha="center", arrowprops=dict(arrowstyle="-", color="black", lw=0.8))
ax.set_xlabel("Number of languages in the dataset")
ax.set_ylabel("Number of countries")
ax.set_title(f"{n_small} of {len(langs_per_country)} countries have 5 or fewer languages")
ax.spines[["top", "right"]].set_visible(False)
fig.savefig("images/languages_distribution.png", dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"Median country: {langs_per_country.median():.0f} languages")
print(f"Countries with 5 or fewer languages: {n_small} of {len(langs_per_country)}")
print(f"Cameroon, Congo and Nigeria together: {top3_langs} of {africa['language'].nunique()} "
      f"languages ({share_top3:.0%})")

Median country: 9 languages
Countries with 5 or fewer languages: 24 of 51
Cameroon, Congo and Nigeria together: 231 of 502 languages (46%)


```{figure} images/languages_distribution.png
:name: fig-langs-distribution
:alt: Histogram of the number of languages per country, with most countries below 20 and Cameroon, Congo and Nigeria far to the right
:width: 85%

Distribution of languages per country across all 51 countries. Most countries
have only a handful of listed languages; three countries are far out on the right.
```

The distribution is very skewed: the median country has 9 listed languages, almost
half of the countries have 5 or fewer, and just three countries cover nearly half (46%) of
all languages in the dataset.

## 3. Does Cameroon stay on top?

Two things could inflate a country's count:

- **Shared languages.** Many languages are listed under several countries
  (Fulani appears in 10). If a country's count is mostly shared languages, it
  isn't very distinctive.
- **The "Congo" row.** The cleaning script merged the Democratic Republic of the
  Congo and the Republic of the Congo into one "Congo". Splitting it could only
  *lower* each part, so it can't overtake Cameroon.

So we split each country's count into languages found **only in that country**
versus languages **shared** with other countries.

In [4]:
countries_per_lang = africa.groupby("language")["country"].nunique()
africa["shared"] = africa["language"].map(countries_per_lang) > 1

split = (
    africa.groupby(["country", "shared"])["language"].nunique()
    .unstack(fill_value=0)
    .rename(columns={False: "only here", True: "shared"})
    .loc[top10_names]
)
split["% shared"] = (split["shared"] / split.sum(axis=1) * 100).round(0).astype(int)

plot = split.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot.index, plot["only here"], color=BLUE, label="Only in this country")
ax.barh(plot.index, plot["shared"], left=plot["only here"], color=GREY,
        label="Shared with other countries")
ax.set_xlabel("Number of languages in the dataset")
ax.set_title("Cameroon still leads when counting only its own languages")
ax.legend(loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.savefig("images/languages_unique_vs_shared.png", dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"Languages found in only one country: {(countries_per_lang == 1).sum()} "
      f"of {len(countries_per_lang)}")
split.sort_values("only here", ascending=False)

Languages found in only one country: 347 of 502


shared,only here,shared,% shared
country,,,
Cameroon,77,18,19
Congo,56,23,29
Nigeria,48,25,34
Sudan,29,11,28
Chad,14,10,42
Burkina Faso,11,25,69
Ghana,9,25,74
South Sudan,9,14,61
Mali,9,13,59


```{figure} images/languages_unique_vs_shared.png
:name: fig-unique-vs-shared
:alt: Stacked bar chart splitting each top-10 country's languages into those found only in that country and those shared with others
:width: 85%

Languages found only in each country (blue) versus shared with other countries
(grey). Cameroon has the most languages of its own (77), and only 19% of its
languages are shared.
```

Cameroon is still first when we count only languages that appear nowhere else
(77, ahead of Congo's 56 and Nigeria's 48). By contrast, countries like Namibia and
Ghana rank high mostly because of cross-border languages.

## 4. What kind of languages drive Cameroon's count?

In [5]:
per_lang = africa.drop_duplicates(["language", "country"])
speakers = per_lang[per_lang["country"].isin(langs_per_country.head(6).index)]
order = langs_per_country.head(6).index

fig, ax = plt.subplots(figsize=(8, 4.5))
data = [speakers.loc[speakers["country"] == c, "native_speakers"] for c in order]
box = ax.boxplot(data, vert=False, labels=order, patch_artist=True,
                 medianprops=dict(color="black"))
for patch, c in zip(box["boxes"], order):
    patch.set_facecolor(ORANGE if c == top_country else BLUE)
    patch.set_alpha(0.7)
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlabel("Native speakers per language (log scale)")
ax.set_title("Cameroon's languages are mostly small (median about 17,000 speakers)")
ax.spines[["top", "right"]].set_visible(False)
fig.savefig("images/speakers_per_language.png", dpi=150, bbox_inches="tight")
plt.close(fig)

summary = per_lang.groupby("country").agg(
    languages=("language", "nunique"),
    median_speakers=("native_speakers", "median"),
).loc[order]
summary["median_speakers"] = summary["median_speakers"].astype(int)
summary

,languages,median_speakers
country,,
Cameroon,95,17000
Congo,79,60000
Nigeria,73,100000
Sudan,40,17000
Burkina Faso,36,200000
Ghana,34,455000


```{figure} images/speakers_per_language.png
:name: fig-speakers
:alt: Box plots of native speakers per language for the six countries with the most languages, on a log scale, with Cameroon's box lowest
:width: 85%

Native speakers per language for the six countries with the most languages (log
scale). A typical Cameroonian language in the dataset has about 17,000 speakers,
compared with about 100,000 in Nigeria. Sudan's languages are similarly small, but
Sudan has far fewer of them.
```

In [6]:
families = (
    africa[africa["country"].isin(order[:5])]
    .groupby(["family", "country"])["language"].nunique()
    .unstack(fill_value=0)[list(order[:5])]
    .sort_values(top_country, ascending=False)
)
families

country,Cameroon,Congo,Nigeria,Sudan,Burkina Faso
family,,,,,
Niger–Congo,92,62,65,22,34
Afroasiatic,2,0,5,1,0
Nilo-Saharan,1,12,3,17,2
Kongo-based,0,1,0,0,0
Ubangian,0,4,0,0,0


Almost all of Cameroon's languages (92 of 95) belong to the **Niger–Congo**
family. So Cameroon's diversity comes from **many small languages within one
family**, not from many different families. Congo and Sudan, by contrast, mix
Niger–Congo with a sizeable group of Nilo-Saharan languages.

## Results

- **Cameroon has the most languages in this dataset (95)**, ahead of "Congo" (79)
  and Nigeria (73). See [](#fig-langs-per-country).
- **The answer is robust.** Cameroon also has the most languages found in no other
  country (77), and splitting the merged "Congo" row could only lower its count.
  See [](#fig-unique-vs-shared).
- **It's driven by many small languages from one family.** Cameroon's languages
  have a median of about 17,000 native speakers, and 92 of 95 are Niger–Congo.
  See [](#fig-speakers).
- **Linguistic diversity is very uneven.** The median country has 9 listed
  languages, and three countries account for 46% of all languages.
  See [](#fig-langs-distribution).

```{warning} Caveats
- **"Congo" is two countries.** The cleaning script merges the Democratic Republic
  of the Congo and the Republic of the Congo into one "Congo" row, so its count is
  inflated.
- **Coverage is uneven.** The dataset lists popular languages only (about 500 of
  Africa's 1,250–2,100+), so the ranking reflects what is in this Wikipedia table,
  not a full census of languages.
- **Speaker counts are rough.** Native-speaker numbers come from Wikipedia and are
  estimates of different ages and quality.
- **Counts are not causes.** These are descriptive counts; they don't explain *why*
  some countries have more languages.
```